<div dir="rtl">

# 06 — عرض توضيحي: `diagnose()`

يحمّل النموذج النهائي (`models/logreg_final.pkl`، مُدرَّب على `train` الكامل في `05_finalize.ipynb`) ويشغّله
على مريض حقيقي من `validate` بثلاثة سيناريوهات: (1) كل الأدلة، (2) خمسة أدلة مُختارة يدوياً، (3) خمسة أدلة
مُختارة بـ`expected_information_gain` — لعرض أثر نقص المعلومات وطريقة اختيار الأدلة على الثقة التشخيصية بشكل
ملموس (نفس الظاهرة المُقاسة كمياً في `03_train.ipynb`/`04_smart_selection.ipynb`، هنا كمثال واحد قابل للقراءة
مباشرة).

**تصحيحات عن مسودة الكود الأصلية**: `label_encoder` اسمه `.joblib` لا `.pkl`، والنموذج لم يكن محفوظاً إطلاقاً
قبل `05_finalize.ipynb` — كلاهما يُحمَّلان الآن من `models/` فعلياً.

</div>

In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
from scipy import sparse

sys.path.insert(0, "..")
from src.selection import ecode, select_evidence_eig

MODELS = Path("../models")
DATA_PROCESSED = Path("../data/processed")

model = joblib.load(MODELS / "logreg_final.pkl")
feature_names = json.load(open(DATA_PROCESSED / "feature_names.json", encoding="utf-8"))
le = joblib.load(MODELS / "label_encoder.joblib")  # الاسم الصحيح .joblib، وليس .pkl

idx = {f: i for i, f in enumerate(feature_names)}
print(f"عدد الميزات: {len(feature_names)} | عدد الأمراض: {len(le.classes_)}")

عدد الميزات: 974 | عدد الأمراض: 49


<div dir="rtl">

## دالة `diagnose()`

تبني متجهاً كثيفاً بطول `feature_names` (974)، تُفعِّل الأعمدة المطابقة لرموز الأدلة المُعطاة (رموز B ثنائية
مباشرة مثل `E_66`، أو رموز C/M بصيغة `الكود_@_القيمة` مثل `E_54_@_V_182` — كلاهما مفتاح مباشر في
`feature_names`)، تضبط `AGE`/`SEX_M`، ثم تستخدم `model.predict_proba` وتطبع أعلى `top` احتمالات.

</div>

In [2]:
def diagnose(evidence_codes, age=30, sex="M", top=5):
    x = np.zeros(len(feature_names))
    for e in evidence_codes:
        if e in idx:
            x[idx[e]] = 1
        else:
            print("تنبيه: رمز غير موجود:", e)
    x[idx["AGE"]] = age
    x[idx["SEX_M"]] = 1 if sex == "M" else 0

    proba = model.predict_proba(sparse.csr_matrix(x.reshape(1, -1)))[0]
    for i in np.argsort(-proba)[:top]:
        print(f"{le.classes_[i]:45s} {proba[i]*100:5.1f}%")

<div dir="rtl">

## مثال على مريض حقيقي: ذكر، 29 سنة، `Anaphylaxis` (من `validate`)

31 دليلاً حقيقياً من `EVIDENCES` الخام للمريض (مزيج من رموز B ثنائية ورموز C/M بصيغة `_@_`)، `INITIAL_EVIDENCE
= E_66`.

</div>

In [3]:
full_evidence = [
    "E_12", "E_42", "E_53", "E_54_@_V_182", "E_55_@_V_84", "E_55_@_V_85", "E_55_@_V_87",
    "E_55_@_V_88", "E_55_@_V_197", "E_56_@_4", "E_57_@_V_123", "E_58_@_1", "E_59_@_7",
    "E_66", "E_129", "E_130_@_V_156", "E_131_@_V_10", "E_132_@_3", "E_133_@_V_26",
    "E_133_@_V_30", "E_133_@_V_31", "E_133_@_V_33", "E_133_@_V_35", "E_134_@_0",
    "E_135_@_V_12", "E_136_@_10", "E_151", "E_152_@_V_89", "E_152_@_V_122", "E_159",
    "E_204_@_V_10",
]
print(f"عدد الأدلة الكاملة: {len(full_evidence)}")

print("\n--- diagnose() بكل الأدلة (31 دليلاً) ---")
diagnose(full_evidence, age=29, sex="M", top=5)

عدد الأدلة الكاملة: 31

--- diagnose() بكل الأدلة (31 دليلاً) ---
Anaphylaxis                                   100.0%
Scombroid food poisoning                        0.0%
Inguinal hernia                                 0.0%
Pancreatic neoplasm                             0.0%
Panic attack                                    0.0%


<div dir="rtl">

## نفس المريض، بخمسة أدلة فقط (`INITIAL_EVIDENCE` + 4 من أوائل القائمة)

</div>

In [4]:
five_evidence = ["E_66", "E_12", "E_42", "E_53", "E_54_@_V_182"]
print("--- diagnose() بخمسة أدلة فقط ---")
diagnose(five_evidence, age=29, sex="M", top=5)

--- diagnose() بخمسة أدلة فقط ---
Anemia                                          8.2%
Anaphylaxis                                     8.2%
Acute dystonic reactions                        6.0%
Myasthenia gravis                               5.9%
Guillain-Barré syndrome                         5.8%


<div dir="rtl">

**الفرق ملموس**: بكامل الأدلة، النموذج **متأكد تماماً** (100.0%) ومُصيب. بخمسة أدلة فقط، `Anaphylaxis`
يتعادل حرفياً مع `Anemia` على المركز الأول (8.2% لكل منهما) — التشخيص الصحيح موجود ضمن الاحتمالات لكنه غير
واضح إطلاقاً، بالضبط النمط الذي قاسته جداول الإسقاط في `03_train.ipynb`/`04_smart_selection.ipynb` كمياً عبر
آلاف المرضى (الدقة تنهار من ~99.7% إلى ~57-78% عند k=3-5). هذا مثال واحد حي على نفس الظاهرة الإحصائية.

</div>

<div dir="rtl">

## نفس المريض، بخمسة أدلة مختارة بـ`expected_information_gain` بدل العشوائي/اليدوي

**`select_evidence_eig` مستوردة من `src/selection.py`** — نفس الدالة المُستخدَمة في `04_smart_selection.ipynb`
حرفياً، لا نسخة مكرَّرة هنا. الجدول `models/nb_evidence_table.npz` (محفوظ في `05_finalize.ipynb`) يُحمَّل
جاهزاً بدل إعادة بنائه. أربع خطوات إضافية بعد `INITIAL_EVIDENCE` = 5 أكواد إجمالاً، كالسيناريو اليدوي أعلاه.

**ملاحظة**: أحد الأكواد المُختارة (`E_152`) متعدد القيم (M) ولهذا المريض تحديداً قيمتان نشطتان معاً
(`E_152_@_V_89` و`E_152_@_V_122`) — فالعناصر الخام الناتجة 6 لا 5، رغم أن عدد **الأكواد** المُختارة 5 بالضبط
(نفس التمييز بين "عدد الأدلة" و"عدد العناصر" الموثَّق سابقاً في `03_train.ipynb`).

</div>

In [5]:
nb = np.load(MODELS / "nb_evidence_table.npz", allow_pickle=True)
P_present = nb["P_present"]
class_prior = nb["class_prior"]
all_codes = nb["all_codes"].tolist()
code_to_idx = {c: i for i, c in enumerate(all_codes)}

items_by_code = {}
for it in full_evidence:
    items_by_code.setdefault(ecode(it), []).append(it)

initial_code = "E_66"
codes = sorted(items_by_code.keys())
print(f"عدد الأكواد المميزة لدى المريض: {len(codes)} | INITIAL_EVIDENCE: {initial_code}")

code_sequence = select_evidence_eig([codes], [initial_code], P_present, class_prior, code_to_idx, k_max=4)
chosen_codes = code_sequence[0]

kept_items_eig = items_by_code[initial_code][:]
for c in chosen_codes:
    kept_items_eig += items_by_code[c]

print(f"\nINITIAL_EVIDENCE (دائماً أول دليل): {initial_code}")
print(f"الأكواد الأربعة الإضافية التي اختارها EIG (بالترتيب): {chosen_codes}")
print(f"الأكواد الخمسة كاملةً: [{initial_code}] + {chosen_codes} = {[initial_code] + chosen_codes}")
print(f"العناصر الخام الناتجة ({len(kept_items_eig)}): {kept_items_eig}")

print("\n--- diagnose() بخمسة أدلة مختارة بـEIG ---")
diagnose(kept_items_eig, age=29, sex="M", top=5)

عدد الأكواد المميزة لدى المريض: 22 | INITIAL_EVIDENCE: E_66

INITIAL_EVIDENCE (دائماً أول دليل): E_66
الأكواد الأربعة الإضافية التي اختارها EIG (بالترتيب): ['E_54', 'E_152', 'E_130', 'E_134']
الأكواد الخمسة كاملةً: [E_66] + ['E_54', 'E_152', 'E_130', 'E_134'] = ['E_66', 'E_54', 'E_152', 'E_130', 'E_134']
العناصر الخام الناتجة (6): ['E_66', 'E_54_@_V_182', 'E_152_@_V_89', 'E_152_@_V_122', 'E_130_@_V_156', 'E_134_@_0']

--- diagnose() بخمسة أدلة مختارة بـEIG ---
Guillain-Barré syndrome                        12.5%
Acute dystonic reactions                       10.6%
Myasthenia gravis                               10.3%
Anaphylaxis                                     8.7%
Ebola                                            7.9%


<div dir="rtl">

**نتيجة صريحة وغير مُجمَّلة**: في هذا المثال الواحد، اختيار `EIG` **أسوأ** من الاختيار اليدوي/العشوائي —
`Anaphylaxis` (التشخيص الصحيح) يتراجع إلى **المركز الرابع (8.7%)**، بينما كان متعادلاً على المركز الأول
(8.2%) مع الاختيار اليدوي. السبب المرجَّح: تبسيط جدول Bayes الساذج (موجود/غائب فقط لكل كود، بلا تمييز القيمة
المحدَّدة C/M — موثَّق كقيد صراحة في `04_smart_selection.ipynb`) اختار أكواداً مفيدة **في المتوسط عبر آلاف
المرضى**، لكن **قيمها المحدَّدة لهذا المريض تحديداً** (`E_54_@_V_182`, `E_152_@_V_89`, ...) صادف أنها مشتركة
مع أمراض عصبية أخرى (Guillain-Barré، Acute dystonic reactions، Myasthenia gravis) أكثر من كونها مميِّزة
لـ`Anaphylaxis` تحديداً.

**الدرس**: نتائج `04_smart_selection.ipynb` (تفوّق `EIG` على `random`/`greedy_entropy`) هي **متوسطات عبر آلاف
المرضى** — لا تضمن تفوّقاً في كل حالة فردية. هذا المثال يُبرِز حدود التبسيط (موجود/غائب فقط) بشكل ملموس، ويؤكد
أهمية عدم الاعتماد على مثال واحد للحكم على استراتيجية اختيار — الجداول الإحصائية الشاملة (لا الأمثلة الفردية)
هي مصدر الحقيقة هنا.

</div>

<div dir="rtl">

## اختبار: رمز دليل غير موجود

</div>

In [6]:
diagnose(["E_9999999"], age=29, sex="M", top=3)

تنبيه: رمز غير موجود: E_9999999
Larygospasm                                    10.6%
Whooping cough                                 10.3%
Guillain-Barré syndrome                         9.9%


<div dir="rtl">

الدالة لا تتعطَّل عند رمز غير معروف — تطبع تنبيهاً وتتجاهله (تشخيص بلا ذلك الدليل، هنا بلا أي دليل فعلي سوى
العمر/الجنس، فالاحتمالات منخفضة ومتقاربة كما هو متوقَّع).

</div>

<div dir="rtl">

## ملخص وخارج النطاق

- `diagnose()` تحمّل `models/logreg_final.pkl` و`models/label_encoder.joblib` فعلياً (كلاهما لم يكونا
  موجودين قبل `05_finalize.ipynb` في هذه المحادثة — المسودة الأصلية أشارت لملفات غير موجودة وباسم خاطئ
  لـ`label_encoder`).
- مثال واقعي واحد (مريض `Anaphylaxis` حقيقي من `validate`) بثلاثة سيناريوهات: **كل الأدلة (31) → 100.0%
  صحيح وواثق**؛ **5 أدلة يدوية → تعادل 8.2% مع Anemia** (التشخيص الصحيح لا يزال الأول لكن غير واضح)؛
  **5 أدلة بـ`expected_information_gain` → Anaphylaxis يتراجع للمركز الرابع (8.7%)**، أسوأ من الاختيار
  اليدوي في هذه الحالة تحديداً — نتيجة صريحة تُبرِز أن تفوّق `EIG` في `04_smart_selection.ipynb` هو **متوسط
  إحصائي عبر آلاف المرضى**، لا ضمانة لكل حالة فردية (تبسيط "موجود/غائب فقط" في جدول Bayes الساذج له ثمن هنا).
- **خارج النطاق عمداً**: لا واجهة تفاعلية (CLI/API) هنا — هذا عرض توضيحي داخل النوتبوك فقط. `greedy_entropy`
  لم يُستخدَم في هذا المثال (يتطلب استدعاءات `predict_proba` حقيقية بدل جدول Bayes المحفوظ). `test` لم يُلمس.
  لا XGBoost.

</div>